# Limpieza de datos agrícolas

**Proyecto:** Visualizador de Productividad Agrícola (V.P. Agrícola)  
**Curso:** Algoritmos y Programación, Universidad de Antioquia

**Fuente:** [Agronet, estadísticas agrícolas](https://agronet.gov.co/estadisticas/agricola)

Este cuaderno muestra cómo se prepara el archivo original para utilizarlo en la aplicación Streamlit. Se conservan los registros originales y se convierten cuatro columnas numéricas a un formato que Python pueda calcular.

## 1. Cargar el archivo original

El archivo `detalle_agricola_nacional.csv` debe estar en `data/raw/`. Este cuaderno se guarda en `data/`, por eso la ruta comienza por `raw/`.

In [1]:
import pandas as pd

df = pd.read_csv("raw/detalle_agricola_nacional.csv")

print("Filas y columnas:", df.shape)
print("Columnas:", df.columns.tolist())
df.head()

Filas y columnas: (30872, 8)
Columnas: ['Año', 'Departamento', 'Producto', 'Área Cosechada', 'Área Sembrada', 'Producción', 'Rendimiento', 'Ciclo']


,Año,Departamento,Producto,Área Cosechada,Área Sembrada,Producción,Rendimiento,Ciclo
0,2025,AMAZONAS,AHUYAMA,"24,00","24,00","66,00","2,75",TRANSITORIO
1,2025,AMAZONAS,AJI,"32,80","33,00","32,80","1,00",TRANSITORIO
2,2025,AMAZONAS,ARAZA,"83,50","94,50","399,75","4,79",PERMANENTE
3,2025,AMAZONAS,ARROZ SECANO MANUAL,"8,00","8,00","6,40","0,80",TRANSITORIO
4,2025,AMAZONAS,ASAI,"158,00","190,00","1.037,00","6,56",PERMANENTE


## 2. Revisar los datos

Se comprueban los tipos de datos y los valores vacíos. No se eliminan filas automáticamente: primero hay que conocer el motivo de cualquier dato faltante.

In [2]:
print(df.dtypes)
print("\nValores vacíos por columna:")
print(df.isna().sum())

Año               int64
Departamento        str
Producto            str
Área Cosechada      str
Área Sembrada       str
Producción          str
Rendimiento         str
Ciclo               str
dtype: object

Valores vacíos por columna:
Año               0
Departamento      0
Producto          0
Área Cosechada    0
Área Sembrada     0
Producción        0
Rendimiento       0
Ciclo             0
dtype: int64


## 3. Convertir las columnas numéricas

En el CSV original, algunos números utilizan el punto como separador de miles y la coma como separador decimal (por ejemplo, `1.037,00`). Primero se quitan los puntos, después se cambian las comas por puntos y finalmente se convierten los valores a números.

`errors="coerce"` transforma en valores vacíos (`NaN`) los textos que no puedan convertirse. Por eso revisamos nuevamente si aparecen valores vacíos.

In [3]:
columnas = ["Área Cosechada", "Área Sembrada", "Producción", "Rendimiento"]

for columna in columnas:
    df[columna] = (
        df[columna]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

print(df.dtypes)
print("\nValores vacíos después de convertir:")
print(df[columnas].isna().sum())
df.head()

Año                 int64
Departamento          str
Producto              str
Área Cosechada    float64
Área Sembrada     float64
Producción        float64
Rendimiento       float64
Ciclo                 str
dtype: object

Valores vacíos después de convertir:
Área Cosechada    0
Área Sembrada     0
Producción        0
Rendimiento       0
dtype: int64


,Año,Departamento,Producto,Área Cosechada,Área Sembrada,Producción,Rendimiento,Ciclo
0,2025,AMAZONAS,AHUYAMA,24.0,24.0,66.00,2.75,TRANSITORIO
1,2025,AMAZONAS,AJI,32.8,33.0,32.80,1.00,TRANSITORIO
2,2025,AMAZONAS,ARAZA,83.5,94.5,399.75,4.79,PERMANENTE
3,2025,AMAZONAS,ARROZ SECANO MANUAL,8.0,8.0,6.40,0.80,TRANSITORIO
4,2025,AMAZONAS,ASAI,158.0,190.0,1037.00,6.56,PERMANENTE


## 4. Comprobar el resultado

Revisamos la cantidad de registros, las columnas y algunos ejemplos. **No eliminamos duplicados**, porque una combinación repetida de producto y departamento podría corresponder a registros legítimos; necesitaríamos verificar la estructura de los datos antes de tomar esa decisión.

In [4]:
print("Cantidad de registros:", len(df))
print("Cantidad de columnas:", len(df.columns))
print("Años disponibles:", sorted(df["Año"].dropna().unique().tolist()))
df.head()

Cantidad de registros: 30872
Cantidad de columnas: 8
Años disponibles: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


,Año,Departamento,Producto,Área Cosechada,Área Sembrada,Producción,Rendimiento,Ciclo
0,2025,AMAZONAS,AHUYAMA,24.0,24.0,66.00,2.75,TRANSITORIO
1,2025,AMAZONAS,AJI,32.8,33.0,32.80,1.00,TRANSITORIO
2,2025,AMAZONAS,ARAZA,83.5,94.5,399.75,4.79,PERMANENTE
3,2025,AMAZONAS,ARROZ SECANO MANUAL,8.0,8.0,6.40,0.80,TRANSITORIO
4,2025,AMAZONAS,ASAI,158.0,190.0,1037.00,6.56,PERMANENTE


## 5. Exportar los datos procesados

Guardamos `data/datos.csv`, que es el archivo utilizado por la aplicación. El archivo original permanece sin modificaciones dentro de `data/raw/`.

In [5]:
df.to_csv("datos.csv", index=False)
print("Archivo guardado: data/datos.csv")

Archivo guardado: data/datos.csv


## Resumen del procedimiento

1. Se cargó el archivo descargado de Agronet.
2. Se revisaron sus columnas, tipos de datos y valores faltantes.
3. Se convirtieron `Área Cosechada`, `Área Sembrada`, `Producción` y `Rendimiento` a valores numéricos.
4. Se verificó la estructura del resultado y se exportó el archivo `datos.csv`.

El objetivo de esta transformación es permitir filtros, sumas, indicadores y gráficos en Streamlit. No se modificaron los nombres de departamentos, productos o ciclos, ni se descartaron registros como parte del procedimiento.